# Lab 6: Pure exploration and learning from preferences

The first part identifies the best arm of a bandit with a fixed budget of pulls: uniform
allocation against sequential halving. The second part ranks models from duels, as in a
chatbot arena: maximum likelihood estimation of the Bradley-Terry model, Elo ratings, and
an active choice of the duels.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

This lab needs the version of `seqlearn` that contains `preferences.py`: if the imports
below fail, download the setup archive again from the course page and replace the folder
`seqlearn/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import BernoulliBandit, UCBAgent, run
from seqlearn import checks
from seqlearn.preferences import Arena, kendall_tau, make_arena, record, sigmoid
from seqlearn.plotting import COLORS, MUTED, set_style

set_style()

# Part I: Pure exploration

## 1. Best-arm identification

A company tests $10$ versions of a web page (an A/B/n test) on a budget of $n$ visitors,
then keeps the version with the best click rate for good. The clicks collected during the
test do not matter: only the final choice does. The criterion is the probability of
recommending a wrong arm, not the regret.

**Question 1.** Complete `uniform_allocation(env, budget)`: it pulls the arms in turn,
$0, 1, \ldots, K-1, 0, 1, \ldots$, until the budget is spent, and returns the arm with the
largest empirical mean (the smallest index in case of a tie).

In [ ]:
def uniform_allocation(env, budget):
    ...  # TODO


checks.check_uniform_allocation(uniform_allocation)

## 2. Sequential halving

Sequential halving (Karnin et al., 2013) splits the budget $n$ into
$L = \lceil \log_2 K \rceil$ phases. In each phase, every remaining arm is pulled
$\lfloor n / (|S| L) \rfloor$ times, where $|S|$ is the number of remaining arms, and the
best half of the arms, $\lceil |S|/2 \rceil$ of them, is kept according to the empirical
means **of this phase only**. The last remaining arm is recommended.

**Question 2.** Complete `sequential_halving(env, budget)`.

*Hint:* `np.argsort(-means, kind="stable")` sorts the arms by decreasing empirical mean,
ties going to the smallest index.

In [ ]:
def sequential_halving(env, budget):
    ...  # TODO


checks.check_sequential_halving(sequential_halving)

**Question 3.** On the instance `MEANS` below, estimate the probability of error of the two
strategies over $200$ runs, for budgets $n \in \{500, 1000, 2000, 4000, 8000\}$, and plot
it as a function of $n$ with a logarithmic y-axis.

In [ ]:
MEANS = [0.5, 0.45, 0.42, 0.4, 0.38, 0.35, 0.3, 0.3, 0.25, 0.2]
BUDGETS = [500, 1000, 2000, 4000, 8000]
N_RUNS = 200

In [ ]:
...  # TODO

# Part II: Learning from duels

## 3. The Bradley-Terry model

An arena hosts $8$ language models. A user asks a question, two models answer, and the user
says which answer is better. In the Bradley-Terry model, each model $i$ has a score $s_i$
and model $i$ beats model $j$ with probability $\sigma(s_i - s_j)$, where
$\sigma(z) = 1/(1 + e^{-z})$. `Arena(scores)` simulates these duels: `arena.duel(i, j)`
returns $1$ if $i$ wins. The scores are unknown in practice; here they are only used to
evaluate the estimates.

The results are stored in a matrix `wins`, where `wins[i, j]` is the number of wins of $i$
against $j$. The log-likelihood of scores $s$ is
$$\ell(s) = \sum_{i \ne j} W_{ij} \log \sigma(s_i - s_j) \enspace,$$
and its gradient is $\partial \ell / \partial s_i = \sum_j \big(W_{ij} - N_{ij}\,
\sigma(s_i - s_j)\big)$, with $N_{ij} = W_{ij} + W_{ji}$: the wins of $i$ minus its
expected wins (Tutorial 6, exercise 2).

**Question 4.** Complete `bt_log_likelihood(scores, wins)` and `bt_gradient(scores, wins)`.

*Hint:* `scores[:, None] - scores[None, :]` is the matrix of the $s_i - s_j$.

In [ ]:
def bt_log_likelihood(scores, wins):
    ...  # TODO


def bt_gradient(scores, wins):
    ...  # TODO


checks.check_bradley_terry(bt_log_likelihood, bt_gradient)

The scores are only defined up to an additive constant, since $\sigma(s_i - s_j)$ does not
change when the same number is added to all the scores. The function below maximizes
$\ell(s) - \frac{\lambda}{2}\|s\|^2$ by gradient ascent: the small penalty fixes the
constant (the scores sum to zero) and keeps them finite when a model has won all its duels.

In [ ]:
def fit_bradley_terry(wins, lam=0.1, n_steps=300):
    scores = np.zeros(len(wins))
    step = 1 / max(1.0, (wins + wins.T).sum(axis=1).max() / 4 + lam)
    for _ in range(n_steps):
        scores += step * (bt_gradient(scores, wins) - lam * scores)
    return scores

**Question 5.** Draw an arena with `make_arena(8, rng=0)`, then $2000$ duels between pairs
chosen uniformly at random (`rng.choice(8, size=2, replace=False)`), stored with `record`.
Fit the Bradley-Terry model, and plot the estimated scores against the true scores, both
centered. Print the Kendall correlation between the two rankings (`kendall_tau`), and the
model with the best estimated score.

In [ ]:
...  # TODO

## 4. Elo ratings

Chess federations and many arenas update ratings after each duel instead of refitting the
model: after a duel between $i$ and $j$, with $y = 1$ if $i$ wins and $0$ otherwise,
$$r_i \leftarrow r_i + k\,\big(y - \sigma(r_i - r_j)\big) \enspace, \qquad
r_j \leftarrow r_j - k\,\big(y - \sigma(r_i - r_j)\big) \enspace.$$
It is a stochastic gradient step on the log-likelihood of this single duel (Tutorial 6,
exercise 2). Chess ratings use the scale $R = 400\, r / \log 10$.

**Question 6.** Complete `elo_update(ratings, i, j, i_wins, k)`, which returns the new
ratings without modifying its argument. Run Elo with $k = 0.05$ on the $2000$ duels of
Question 5, in their order, and print its Kendall correlation with the true scores. Then
run it again on the same duels in $5$ random orders, and compare the final ratings of
model $0$ with its Bradley-Terry score.

In [ ]:
def elo_update(ratings, i, j, i_wins, k):
    ...  # TODO


checks.check_elo(elo_update)

In [ ]:
...  # TODO

## 5. Choosing the duels

Duels cost money (each one is a human annotation). To find the best model, it is wasteful
to keep comparing the weakest ones. **Halving on duels** applies sequential halving to the
models: in each of the $L = \log_2 8 = 3$ phases, every pair of remaining models duels
$\lfloor n / (L \cdot \text{number of pairs}) \rfloor$ times, the Bradley-Terry model is
fitted on all the duels so far, and the best half of the remaining models is kept.

**Question 7.** Implement `uniform_duels(arena, budget, rng)`, which spends the budget on
uniformly random pairs and returns the model with the best fitted score, and
`halving_duels(arena, budget)`. Estimate the probability of finding the best model over
$200$ arenas `make_arena(8, rng=seed)`, for budgets $\{100, 200, 400, 800\}$, and plot it.

In [ ]:
...  # TODO

## Going further

**Question 8.** Three models play rock-paper-scissors: $0$ beats $1$, $1$ beats $2$, and
$2$ beats $0$, each time with probability $0.8$. Simulate $300$ duels per pair, fit the
Bradley-Terry model and comment. Which model would you call the best?

In [ ]:
...  # TODO